# Decoding Strategies — runnable teaching notebook

Turning a model's next-token distribution into text. This notebook is the executable companion to the [Decoding Strategies](../17-Decoding-Strategies.md) note: every number on the page is computed here, and the figures are regenerated from the **same** functions in `decoding_strategies.py`. One idea per cell; each cell **asserts its claim before printing**, so a green run is a proof that the page's numbers are right.

It is the *general sequence-generation* view of decoding — **beam search** for closed-ended tasks (machine translation, summarization) and **sampling** for open-ended generation. The LLM-systems angle (KV-cache, speculative decoding throughput) lives in the sibling chapter *09. LLMs / 18-Decoding-and-Sampling* and is not repeated here.

Runs on CPU in seconds. The core demos are pure NumPy (fully deterministic); the final cell optionally reproduces neural text degeneration on GPT-2 if `transformers` is installed.

In [1]:
# All decoding logic lives in ONE seeded source of truth: decoding_strategies.py.
# The page, this notebook, and make_figures_17.py all import from it, so no number can drift.
import platform
import numpy as np

import decoding_strategies as ds

print(f"python {platform.python_version()}   numpy {np.__version__}   device: cpu (pure-numpy, deterministic)")

python 3.12.13   numpy 2.4.6   device: cpu (pure-numpy, deterministic)


## 1. The problem: generation is search over an exponential space

A model gives a distribution over the next token; a *sequence* probability is the chain-rule product of per-step conditionals. The number of length-$T$ sequences is $|V|^T$ — astronomically large, so exact $\arg\max$ over sequences is intractable and every decoder is a heuristic.

In [2]:
# Just feel the size of the search: |V|^T candidate sequences.
vocab_size, length = 50_000, 20
num_sequences = vocab_size ** length
assert num_sequences > 1e90, "the sequence space must be astronomically large"
print(f"|V|={vocab_size:,}, T={length}  ->  {num_sequences:.1e} possible sequences")
print("(more than the ~1e80 atoms in the observable universe — we never enumerate this)")

|V|=50,000, T=20  ->  9.5e+93 possible sequences
(more than the ~1e80 atoms in the observable universe — we never enumerate this)


## 2. Greedy is myopic; beam search recovers the globally best sequence

On the two-step tree, greedy takes the locally-best first token **A** and ends at **AX** (joint 0.22). The genuinely most-probable sequence is **BP** (0.4275), reachable only by keeping the *lower*-probability first token B alive — which beam search (width 2) does.

In [3]:
g_seq, g_p = ds.greedy_two_step()
beam_seq, beam_p = ds.beam_two_step(beam_width=2)
best_seq, best_p = ds.best_two_step()

# Assert the point BEFORE printing it: beam must beat greedy and match the global best.
assert (g_seq, round(g_p, 4)) == ("AX", 0.22), "greedy ends at AX (0.22)"
assert (beam_seq, round(beam_p, 4)) == ("BP", 0.4275), "beam width 2 recovers BP (0.4275)"
assert beam_p > g_p and (beam_seq, round(beam_p, 4)) == (best_seq, round(best_p, 4))

print(f"greedy        -> {g_seq}  joint p = {g_p:.4f}")
print(f"beam (B=2)    -> {beam_seq}  joint p = {beam_p:.4f}")
print(f"global best   -> {best_seq}  joint p = {best_p:.4f}")
print(f"\nbeam's BP nearly DOUBLES greedy's AX ({beam_p/g_p:.2f}x); greedy never considered B.")

greedy        -> AX  joint p = 0.2200
beam (B=2)    -> BP  joint p = 0.4275
global best   -> BP  joint p = 0.4275

beam's BP nearly DOUBLES greedy's AX (1.94x); greedy never considered B.


### Why log-space? Beam scores are sums of log-probs
Multiplying hundreds of probabilities underflows to 0.0 in float; summing their logs is numerically stable and monotonic in the product, so it preserves the ranking. The figure `decode_beam_trace.png` traces this prune step on the `{a,b,c}` tree.

In [4]:
# A product of many small probabilities underflows; the log-sum does not.
# float32 is the realistic case for a model's activations: 0.3**200 is already 0.0 there,
# so EVERY hypothesis would tie at zero and the comparison breaks. The sum of logs stays finite.
n_tokens = 200
raw_product_f32 = np.prod(np.full(n_tokens, 0.3, dtype=np.float32))  # underflows to exactly 0.0
log_sum = float(np.sum(np.log(np.full(n_tokens, 0.3))))             # stable, monotonic in the product
assert raw_product_f32 == np.float32(0.0), "a 200-term product of 0.3 underflows to 0.0 in float32"
assert np.isfinite(log_sum), "the sum of logs stays finite and comparable"
print(f"product of 200x 0.3 (float32) = {raw_product_f32}   (underflowed — every hypothesis ties at 0!)")
print(f"sum of logs                   = {log_sum:.2f}   (stable — this is why beam scores in log-space)")

product of 200x 0.3 (float32) = 0.0   (underflowed — every hypothesis ties at 0!)
sum of logs                   = -240.79   (stable — this is why beam scores in log-space)


## 3. Length normalization: don't punish long sequences

Every log-prob is negative, so a longer hypothesis **always** has a smaller raw sum — even when its per-token quality is *higher*. Raw scoring crowns the short one; the length-fair per-token **mean** correctly crowns the higher-quality long one. The Google-NMT $\text{lp}(y)$ is a softened, partial version (its $+5$ constant keeps even $\alpha=1$ from fully averaging).

In [5]:
short_logps = [np.log(0.55), np.log(0.55)]   # 2 mediocre tokens (p=0.55 each)
long_logps = [np.log(0.70)] * 5              # 5 genuinely-better tokens (p=0.70 each)

raw_short, raw_long = float(np.sum(short_logps)), float(np.sum(long_logps))
mean_short, mean_long = ds.mean_log_prob(short_logps), ds.mean_log_prob(long_logps)

assert raw_short > raw_long, "raw sum is biased toward the SHORT sequence"
assert mean_long > mean_short, "the length-fair per-token mean prefers the higher-quality LONG one"

print(f"raw sum-of-log-probs : short {raw_short:.3f}  long {raw_long:.3f}  -> winner: "
      f"{'short' if raw_short > raw_long else 'long'}")
print(f"mean log-prob (fair) : short {mean_short:.3f}  long {mean_long:.3f}  -> winner: "
      f"{'short' if mean_short > mean_long else 'long'}")
print("\nGoogle-NMT lp() partial correction (sum / lp(len)) at three alphas:")
for alpha in (0.0, 0.7, 1.0):
    s = ds.normalized_beam_score(short_logps, alpha)
    long_score = ds.normalized_beam_score(long_logps, alpha)
    print(f"   alpha={alpha:>3}: short {s:.3f}  long {long_score:.3f}")

raw sum-of-log-probs : short -1.196  long -1.783  -> winner: short
mean log-prob (fair) : short -0.598  long -0.357  -> winner: long

Google-NMT lp() partial correction (sum / lp(len)) at three alphas:
   alpha=0.0: short -1.196  long -1.783
   alpha=0.7: short -1.073  long -1.247
   alpha=1.0: short -1.025  long -1.070


## 4. Temperature reshapes the softmax (measured via entropy)

$p_i = \mathrm{softmax}(z_i / T)$. Dividing logits by $T$ rescales every gap by $1/T$: $T<1$ sharpens (toward greedy), $T>1$ flattens (toward uniform). Entropy $H = -\sum p_i \log_2 p_i$ puts a number on the peakiness.

In [6]:
logits = np.array(ds.TEMP_LOGITS)   # [3, 2, 1, 0] for tokens A, B, C, D
entropies = {}
for T in ds.TEMPERATURES:
    p = ds.softmax_with_temperature(logits, T)
    h = ds.entropy_bits(p)
    entropies[T] = h
    print(f"T={T:>3}:  [{'  '.join(f'{x:.3f}' for x in p)}]   H={h:.3f} bits")

# Contract: hotter temperature => flatter => strictly higher entropy.
assert entropies[0.5] < entropies[1.0] < entropies[2.0], "entropy must rise with T"
assert abs(ds.softmax_with_temperature(logits, 1.0)[0] - 0.644) < 1e-3   # raw top token 0.644
assert abs(ds.softmax_with_temperature(logits, 0.5)[0] - 0.865) < 1e-3   # sharpened to 0.865
print(f"\nentropy rises {entropies[0.5]:.2f} -> {entropies[1.0]:.2f} -> {entropies[2.0]:.2f} bits "
      f"as T goes 0.5 -> 1 -> 2 (figure: decode_temperature_softmax.png)")

T=0.5:  [0.865  0.117  0.016  0.002]   H=0.657 bits
T=1.0:  [0.644  0.237  0.087  0.032]   H=1.367 bits
T=2.0:  [0.455  0.276  0.167  0.102]   H=1.796 bits

entropy rises 0.66 -> 1.37 -> 1.80 bits as T goes 0.5 -> 1 -> 2 (figure: decode_temperature_softmax.png)


## 5. Truncation: top-k (fixed count) vs top-p / nucleus (adaptive mass)

On the **same** 8-token distribution, top-k=2 keeps exactly 2 tokens (mass 0.65) regardless of shape; top-p=0.9 keeps the smallest set whose cumulative mass $\ge 0.9$ — here 5 tokens (mass 0.92). Top-p *adapts*; top-k cannot.

In [7]:
dist = np.array(ds.EIGHT_TOKEN_DIST)
k_keep = ds.top_k_keep(dist, ds.TOP_K)
p_keep = ds.top_p_keep(dist, ds.TOP_P)

assert int(k_keep.sum()) == 2 and abs(dist[k_keep].sum() - 0.65) < 1e-9
assert int(p_keep.sum()) == 5 and abs(dist[p_keep].sum() - 0.92) < 1e-9

print(f"top-k (k={ds.TOP_K}): keeps {int(k_keep.sum())} tokens, mass {dist[k_keep].sum():.2f}")
print(f"top-p (p={ds.TOP_P}): keeps {int(p_keep.sum())} tokens, mass {dist[p_keep].sum():.2f}")

# Renormalize the nucleus: relative odds preserved, survivors sum to 1.
renorm = ds.renormalize(dist, p_keep)
assert abs(renorm.sum() - 1.0) < 1e-9
assert abs(renorm[0] / renorm[1] - dist[0] / dist[1]) < 1e-9, "relative odds preserved"
print(f"renormalized survivors: [{'  '.join(f'{x:.3f}' for x in renorm[p_keep])}]  (sum {renorm.sum():.1f})")

top-k (k=2): keeps 2 tokens, mass 0.65
top-p (p=0.9): keeps 5 tokens, mass 0.92
renormalized survivors: [0.435  0.272  0.141  0.087  0.065]  (sum 1.0)


### The adaptivity, directly: nucleus size on a peaked vs flat distribution
Top-p keeps **few** tokens when the model is confident (peaked) and **many** when it is uncertain (flat). Top-k keeps the same count either way. This is the whole reason nucleus sampling beats fixed-k (figure: `decode_nucleus_adapts.png`).

In [8]:
peaked = ds.softmax(np.array(ds.PEAKED_LOGITS))
flat = ds.softmax(np.array(ds.FLAT_LOGITS))
n_peaked, n_flat = ds.nucleus_size(peaked, ds.TOP_P), ds.nucleus_size(flat, ds.TOP_P)

assert n_peaked < n_flat, "nucleus must be smaller on a peaked distribution than a flat one"
assert int(ds.top_k_keep(peaked, 5).sum()) == int(ds.top_k_keep(flat, 5).sum()) == 5
print(f"top-p={ds.TOP_P} nucleus:  {n_peaked} tokens on PEAKED dist,  {n_flat} on FLAT dist  (adapts)")
print(f"top-k=5      :  5 tokens on BOTH (fixed — ignores the distribution's shape)")

top-p=0.9 nucleus:  1 tokens on PEAKED dist,  9 on FLAT dist  (adapts)
top-k=5      :  5 tokens on BOTH (fixed — ignores the distribution's shape)


## 6. Neural text degeneration: greedy loops, sampling escapes

On a toy self-reinforcing transition model (after the loop token the loop token is most probable — the pattern-continuation pressure Holtzman et al. identify), greedy walks straight into the `I love pizza .` loop. distinct-2 (fraction of unique bigrams) collapses; sampling stays varied. **Same model — only the decoding changes** (figure: `decode_degeneration.png`).

In [9]:
greedy_ids = ds.generate_greedy()
lowt_ids = ds.generate_sampled(temperature=0.3)
samp_ids = ds.generate_sampled(temperature=1.0)
d_greedy, d_lowt, d_samp = (ds.distinct_rate(x) for x in (greedy_ids, lowt_ids, samp_ids))

assert d_greedy < d_samp, "greedy must be MORE repetitive (lower distinct-2) than T=1 sampling"
assert d_lowt < d_samp, "low temperature is more repetitive than full-temperature sampling"

def show(ids):
    return ' '.join(ds.DEGEN_VOCAB[i] for i in ids if ds.DEGEN_VOCAB[i] != '<s>')[:60]

print(f"greedy         distinct-2 = {d_greedy:.3f}  ->  {show(greedy_ids)}...")
print(f"sample (T=0.3) distinct-2 = {d_lowt:.3f}  ->  {show(lowt_ids)}...")
print(f"sample (T=1.0) distinct-2 = {d_samp:.3f}  ->  {show(samp_ids)}...")
print(f"\ndecoding ALONE moves distinct-2 from {d_greedy:.2f} (loop) to {d_samp:.2f} (varied)")

greedy         distinct-2 = 0.125  ->  I love pizza . I love pizza . I love pizza . I love pizza . ...
sample (T=0.3) distinct-2 = 0.175  ->  I love pizza and pizza . I love pizza . I love pizza . I lov...
sample (T=1.0) distinct-2 = 0.450  ->  I love pizza and cake too . I love cake too I and I love piz...

decoding ALONE moves distinct-2 from 0.12 (loop) to 0.45 (varied)


## 7. The same effect on a real model (GPT-2)

The toy model above is the *mechanism*; here it is on real weights. Feed GPT-2 a repetitive prompt: **greedy loops** (distinct-2 ≈ 0.10) while **nucleus sampling stays varied** (distinct-2 ≈ 1.0). Same checkpoint, same prompt — the decoding strategy alone is the difference. (Skips cleanly if `transformers` isn't installed.)

In [10]:
try:
    import torch
    from transformers import GPT2LMHeadModel, GPT2TokenizerFast

    tok = GPT2TokenizerFast.from_pretrained("gpt2")
    model = GPT2LMHeadModel.from_pretrained("gpt2").eval()
    torch.manual_seed(0)

    def distinct_2(text):
        toks = text.split()
        grams = [tuple(toks[i:i + 2]) for i in range(len(toks) - 1)]
        return len(set(grams)) / max(len(grams), 1)

    enc = tok("I love pizza. I love pizza.", return_tensors="pt")
    ids, attn = enc.input_ids, enc.attention_mask
    with torch.no_grad():
        greedy = model.generate(ids, attention_mask=attn, max_new_tokens=40,
                                do_sample=False, pad_token_id=tok.eos_token_id)
        nucleus = model.generate(ids, attention_mask=attn, max_new_tokens=40, do_sample=True,
                                 top_p=0.92, top_k=0, temperature=1.0,
                                 pad_token_id=tok.eos_token_id)
    g = tok.decode(greedy[0, ids.shape[1]:], skip_special_tokens=True)
    n = tok.decode(nucleus[0, ids.shape[1]:], skip_special_tokens=True)
    dg, dn = distinct_2(g), distinct_2(n)
    assert dg < dn, "greedy must be more repetitive than nucleus on this prompt"
    print(f"GREEDY  distinct-2 = {dg:.3f} -> {g[:80]!r}")
    print(f"NUCLEUS distinct-2 = {dn:.3f} -> {n[:80]!r}")
except ImportError:
    print("transformers not installed — skipping the GPT-2 demo (the toy model above shows the same effect).")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GREEDY  distinct-2 = 0.103 -> ' I love pizza. I love pizza. I love pizza. I love pizza. I love pizza. I love pi'
NUCLEUS distinct-2 = 1.000 -> ' I love pizza.\n\n...'


## Recap

- A model only outputs a next-token distribution; the **decoding strategy** turns it into text.
- **Search** (greedy, beam) chases the most-probable sequence — right for closed-ended tasks (MT, summarization), but **degenerates** (bland, repetitive) on open-ended text.
- **Sampling** fixes that: **temperature** reshapes the softmax, **top-k** truncates to a fixed count, **top-p / nucleus** truncates to an *adaptive* mass.
- Everything is one navigation of the **quality–diversity trade-off**.

Every figure on the page is regenerated from these same functions by `make_figures_17.py`. See the [references](../17-Decoding-Strategies.references.md) for the suggested learning path.